# ML11 · VAE 变分自编码器

把上一本的自编码器（autoencoder）升级成生成模型（generative model）：核心观念是「把输入编码成一个分布、再从分布采样」，让潜在空间变得连续、可生成全新数据。

## 学习目标
- 说明变分自编码器（variational autoencoder, VAE）与一般自编码器（autoencoder, AE）的核心差异：编码成「分布」而非「单一点」。
- 理解编码器输出的是分布参数：平均（mu）与对数变异（logvar, log variance）。
- 掌握重参数化技巧（reparameterization trick），知道它为何能让采样过程可微分、可做反向传播（backpropagation）。
- 创建 KL 散度（KL divergence, Kullback-Leibler divergence）的直觉，了解总损失等于重建损失（reconstruction loss）加 KL 散度，以及两者的拉锯。
- 学会从潜在空间（latent space）采样，生成训练数据中没出现过的新样本。

In [ ]:
# 概念：统一导入套件并设置画图字体，让中文标题不会变成方框
import numpy as np
import matplotlib.pyplot as plt

# 技巧：指定支持中文的字体，并修正负号显示，避免图上中文与负号变成乱码
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(0)   # 固定乱数种子，让每次运行的采样结果可重现
print("环境设置完成")

## 从 AE 到 VAE：为什么要编码成分布

自编码器（AE）把输入压成潜在空间（latent space）中的「一个点」，再从那个点还原回去。问题是这些点之间是空洞、不连续的：在两点中间随便取一个位置解码，往往得到无意义的结果，所以 AE 难以拿来「生成」新数据。

VAE 改成把每个输入编码成「一团分布」而非一个点，这就是确定性编码（deterministic encoding）与几率编码（probabilistic encoding）的差别。一团分布会覆盖一片连续区域，让相邻的点也有意义，潜在空间因此变得连续、可采样，这是能生成新数据的关键。

为什么先讲这个：先创建「AE 是点、VAE 是分布」的对比，后面的 mu、logvar、采样才有依归。

In [ ]:
# 概念：对比 AE 把数据压成「离散的点」、VAE 把数据压成「连续的一团分布」

# 造一批二维「建筑特征点」当示范：每点是一栋建筑的 (楼高, 面积) 在潜在空间的位置
n = 6
ae_points = rng.uniform(low=-2.0, high=2.0, size=(n, 2))   # AE：每栋建筑对应潜在空间的一个点

# VAE：同样 n 栋建筑，但每栋是一团分布（以该点为中心、四周散开一片）
cloud_per_point = 80                                       # 每团画几个采样点来表现「一片区域」
vae_clouds = []
for center in ae_points:
    # 在中心附近洒一圈点，示意「这栋建筑覆盖潜在空间的一片连续区域」
    cloud = center + rng.normal(loc=0.0, scale=0.45, size=(cloud_per_point, 2))
    vae_clouds.append(cloud)
vae_clouds = np.vstack(vae_clouds)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(ae_points[:, 0], ae_points[:, 1], c="crimson", s=60)
axes[0].set_title("AE：每栋压成一个点（点与点之间空洞）")
axes[1].scatter(vae_clouds[:, 0], vae_clouds[:, 1], c="steelblue", s=8, alpha=0.4)
axes[1].set_title("VAE：每栋压成一团分布（覆盖连续区域）")
for ax in axes:
    ax.set_xlabel("潜在维度 1"); ax.set_ylabel("潜在维度 2")
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
plt.tight_layout(); plt.show()

print("AE 只有", n, "个离散点；VAE 用", vae_clouds.shape[0], "个采样点铺成连续区域")

## 编码器输出：mu 与 logvar

VAE 的编码器（encoder）不直接输出潜在矢量，而是对每个潜在维度输出两个数：平均 mu 与对数变异 logvar。这等于在描述「这个输入对应到潜在空间的哪一团（mu 是中心）、有多分散（变异越大越散）」。

为何输出 logvar 而不是直接输出变异（variance）：变异必须为正，但神经网络的输出可正可负；改成让网络输出 logvar，再用指数还原变异，就保证为正且数值稳定。

公式：sigma = exp(0.5 × logvar)，其中 sigma 是标准差（standard deviation），描述这团常态分布（normal distribution）的宽度。

In [ ]:
# 概念：编码器把输入转成每个潜在维度的 mu 与 logvar，再由 logvar 还原 sigma

# 造一笔小输入（一栋建筑的标准化特征）当示范用
x = np.array([0.8, -0.3, 1.2])

# 假想编码器：用固定权重把 3 维输入线性投影成 2 维潜在分布的 mu 与 logvar
# 注意：真实 VAE 这些权重是学出来的；这里写死只为了让「参数即一团分布」可手算观察
W_mu = np.array([[0.5, -0.2, 0.1], [0.0, 0.4, -0.3]])
W_logvar = np.array([[-0.1, 0.2, 0.0], [0.3, -0.1, 0.2]])

mu = W_mu @ x            # 这团分布在 2 维潜在空间的中心
logvar = W_logvar @ x    # 对数变异；网络直接输出它以保证还原出的变异为正

sigma = np.exp(0.5 * logvar)   # 由 logvar 还原标准差：sigma = exp(0.5 * logvar)

print("输入 x:", x)
print("mu     (分布中心):", np.round(mu, 3))
print("logvar (对数变异):", np.round(logvar, 3))
print("sigma  (分散程度):", np.round(sigma, 3))   # 全为正，代表每个维度的展开宽度

## 重参数化技巧 reparameterization trick

要训练 VAE 就得从编码器给的分布里「采样（sampling）」出潜在矢量 z。但直接从分布随机抽样是不可微的：随机性卡在计算路径中间，梯度无法穿过它，反向传播（backpropagation）就断了。

重参数化技巧（reparameterization trick）把随机性外移：固定从标准常态抽一个杂讯 epsilon，再用 z = mu + sigma × epsilon 组出样本。如此 mu、sigma 留在可微分的主干上，epsilon 只是外部输入，梯度就能顺着 mu、sigma 通过。这是 VAE 能用梯度下降（gradient descent）训练的核心技巧。

形状：z = mu + sigma × epsilon，epsilon ~ 标准常态 N(0, 1)。

In [ ]:
# 概念：固定 mu、sigma，反复套用 z = mu + sigma * epsilon，观察采样点形成一团

# 沿用「一团分布」的参数当示范（二维潜在空间）
mu = np.array([1.0, -0.5])       # 分布中心
sigma = np.array([0.6, 0.3])     # 两个维度各自的分散程度

n_draw = 300
epsilon = rng.normal(loc=0.0, scale=1.0, size=(n_draw, 2))   # 随机性的唯一来源，取自标准常态
z = mu + sigma * epsilon          # 重参数化：把随机 epsilon 缩放平移成这团分布的样本

print("采样点 z 的实际平均:", np.round(z.mean(axis=0), 3), "（应接近 mu）")
print("采样点 z 的实际标准差:", np.round(z.std(axis=0), 3), "（应接近 sigma）")

plt.figure(figsize=(5, 5))
plt.scatter(z[:, 0], z[:, 1], s=10, alpha=0.4, c="steelblue")
plt.scatter(mu[0], mu[1], c="crimson", s=80, marker="x", label="mu 中心")
plt.title("重参数化采样：z = mu + sigma * epsilon")
plt.xlabel("潜在维度 1"); plt.ylabel("潜在维度 2"); plt.legend()
plt.axis("equal"); plt.show()

## KL 散度：把潜在分布拉回标准常态

KL 散度（KL divergence）衡量「两个分布有多不同」。在 VAE 里它衡量「编码器产生的分布」离「标准常态分布 N(0,1)」有多远，这个标准常态就是我们对潜在空间的先验（prior）假设。

把 KL 当成损失的一部分，就是一种正则化（regularization）：它要求每个输入的潜在团都靠拢在原点附近、彼此不要散得太开，潜在空间才会紧凑又连续，方便日后采样生成。

VAE 用的封闭式（每个维度相加）：KL = -0.5 × Σ(1 + logvar − mu² − exp(logvar))。当 mu 接近 0、logvar 接近 0（即变异接近 1）时，KL 接近 0，代表已贴近标准常态。

In [ ]:
# 概念：用简化 KL 封闭式，比较「贴近标准常态」与「偏离很远」两种分布的 KL 值

def kl_divergence(mu, logvar):
    # 对每个潜在维度算 KL 再相加；mu、logvar 越接近 0，这团越像标准常态，KL 越小
    return -0.5 * np.sum(1 + logvar - mu**2 - np.exp(logvar))

# 造三组分布参数做对照（皆自造）
near = (np.array([0.05, -0.03]), np.array([0.02, -0.01]))   # 几乎就是标准常态
far_center = (np.array([3.0, -2.5]), np.array([0.0, 0.0]))  # 中心偏离原点很远
too_wide = (np.array([0.0, 0.0]), np.array([2.5, 2.5]))     # 变异过大、分布太宽

for name, (mu, logvar) in [("贴近标准常态", near), ("中心偏离很远", far_center), ("分布太宽", too_wide)]:
    print(f"{name:6s}  mu={mu}  logvar={logvar}  ->  KL = {kl_divergence(mu, logvar):.3f}")

# 注意：KL 永远 >= 0，等于 0 只在分布完全等于标准常态时发生

## 总损失：重建损失与 KL 的拉锯

VAE 的总损失由两项相加：重建损失（reconstruction loss）要求解码器「还原得像」原输入，常用均方误差（mean squared error, MSE）或交叉熵（cross-entropy）；KL 散度则要求「潜在空间规矩」。

两项方向相反、互相拉扯（trade-off）：只顾重建会让潜在团各自乱跑、空间不连续；只顾 KL 会让所有输入挤成一团、还原得很糊。这个平衡正是为什么 VAE 生成的样本通常较平滑但略模糊。实务上常加一个权重 beta 调整 KL 的比重（总损失 = 重建 + beta × KL）。

对照：二元交叉熵的极简式为 −Σ[ x·log(x̂) + (1−x)·log(1−x̂) ]，适用于输出介于 0 到 1 的数据。

In [ ]:
# 概念：手算一笔样本的重建损失与 KL，相加成总损失，并调整 beta 看哪一项主导

# 造一笔样本：原输入 x 与解码器还原出的 x_hat（自造，假设还原得还算接近）
x = np.array([0.8, 0.2, 0.6])
x_hat = np.array([0.7, 0.35, 0.55])
mu = np.array([1.5, -1.2])
logvar = np.array([0.1, -0.2])

recon_loss = np.mean((x - x_hat) ** 2)                              # 重建损失：用 MSE 衡量还原误差
kl = -0.5 * np.sum(1 + logvar - mu**2 - np.exp(logvar))             # KL 散度（沿用封闭式）

for beta in [1.0, 5.0, 20.0]:
    total = recon_loss + beta * kl                                 # 总损失：重建 + beta * KL
    kl_share = (beta * kl) / total                                 # KL 项占总损失的比例
    print(f"beta={beta:4.1f}  重建={recon_loss:.4f}  KL={kl:.4f}  总损失={total:.4f}  KL 占比={kl_share:.1%}")

# 技巧：beta 越大，KL 对总损失的影响越重，潜在空间更规矩但重建会更模糊（beta-VAE 的概念）

## 从潜在空间采样以生成新样本

训练完成后，编码器其实可以丢掉：直接在潜在空间从先验（标准常态）随机取一个点 z，交给解码器（decoder），就能生成训练集里没出现过的新样本，这就是生成（generation）与重建（reconstruction）的差别——重建要先有输入，生成则无中生有。

更进一步，在两个潜在点之间做插值（latent space interpolation），逐一解码，会看到生成样本「平滑变形」。这正验证了潜在空间确实连续：相邻的位置对应相似的样本。

形状：插值 z(t) = (1 − t) × A + t × B，t 从 0 到 1。

In [ ]:
# 概念：在潜在空间随机取点与沿一条路径插值，喂入假想解码器，观察生成特征连续变化

def decoder(z):
    # 假想解码器：把 2 维潜在矢量映射回两个「建筑特征」（容积率、楼高）
    # 真实 VAE 的解码器是学出来的；这里写死成平滑函数，方便看出潜在空间的连续性
    far = 1.5 + 0.8 * z[0] + 0.3 * z[1]        # 容积率（floor area ratio）
    height = 20.0 + 6.0 * z[0] - 2.0 * z[1]    # 楼高（公尺）
    return np.array([far, height])

# 生成：直接从标准常态取 5 个全新潜在点并解码（不需要任何输入）
print("从先验随机生成的新地块:")
for z in rng.normal(size=(5, 2)):
    far, height = decoder(z)
    print(f"  z={np.round(z, 2)}  ->  容积率={far:.2f}  楼高={height:.1f} 公尺")

# 插值：在低密度点 A 与高密度点 B 之间取等距点，逐一解码
A = np.array([-1.5, 0.5])
B = np.array([2.0, -1.0])
ts = np.linspace(0, 1, 6)                       # 0 到 1 的等距插值系数
decoded = np.array([decoder((1 - t) * A + t * B) for t in ts])

plt.figure(figsize=(6, 4))
plt.plot(ts, decoded[:, 0], "o-", label="容积率")
plt.plot(ts, decoded[:, 1], "s-", label="楼高(公尺)")
plt.title("潜在空间插值：A 到 B 之间特征平滑过渡")
plt.xlabel("插值系数 t（0=A, 1=B）"); plt.ylabel("生成特征值"); plt.legend()
plt.tight_layout(); plt.show()

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）用 mu 与 sigma 采样一团潜在点（集成：mu/logvar + 重参数化）
#   情境：某社区一栋建筑被编码成潜在空间里的一团分布，给定 mu 与 logvar（数据自造）。
#   要求：
#     1. 自造二维的 mu 与 logvar，由 logvar 还原 sigma，印出这团分布的中心与分散程度。
#     2. 用重参数化 z = mu + sigma * epsilon 采样 200 个潜在点（epsilon 取自标准常态），画出散点。
#   验收：应看到一团以 mu 为中心、宽度由 sigma 决定的点云。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）计算一笔样本的 VAE 总损失（集成：mu/logvar + KL + 重建损失 + 总损失）
#   情境：一笔自造的都市地块特征矢量（楼高、面积、容积率）经过假想 VAE，得到 mu、logvar 与重建输出（数据自造）。
#   要求：
#     1. 用简化 KL 封闭式算出 KL 散度。
#     2. 用均方误差（MSE）算出重建损失。
#     3. 相加得到总损失，再改用一个较大的 beta 重算，比较哪一项变得主导。
#   验收：应看到总损失随 beta 增大时，KL 项对总损失的影响明显上升。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）在潜在空间插值生成连续变化的地块（集成：采样 + 解码器 + 插值 + 潜在空间连续性）
#   情境：手上有两种都市地块各自对应的潜在点 A、B（例如低密度与高密度，数据自造），想生成介于两者之间的过渡型态。
#   要求：
#     1. 写一个假想解码器函数，把潜在矢量映射回地块特征（容积率、楼高）。
#     2. 在 A 与 B 之间取若干等距插值点（用 z(t) = (1-t)*A + t*B），逐一解码。
#     3. 观察生成特征是否随插值平滑变化，并简短说明这代表潜在空间的什么性质。
#   验收：应看到地块特征沿插值路径连续且单调地过渡，印证潜在空间的连续性。
# TODO: 在这里完成


## 小结

- VAE 与 AE 的核心差异：AE 把输入压成潜在空间中的一个点，VAE 压成一团分布，使潜在空间连续、可采样，才能用来生成新数据。
- 编码器输出分布参数 mu 与 logvar；改输出 logvar 是为了保证还原出的变异为正且数值稳定，sigma = exp(0.5 × logvar)。
- 重参数化技巧用 z = mu + sigma × epsilon 把随机性外移到 epsilon，让 mu、sigma 可微分，梯度才能通过、能用梯度下降训练。
- KL 散度衡量编码分布离标准常态先验有多远，当成正则化把潜在团拉回原点附近，使空间紧凑连续。
- 总损失 = 重建损失 + beta × KL；重建要求像、KL 要求规矩，两者拉锯，也解释了 VAE 生成样本平滑但略模糊的特性。
- 生成时丢掉编码器，直接从先验采样交给解码器即可造出新样本；在两点间插值可看到样本平滑变形，印证潜在空间连续。